In [1]:
import sys
# TO CHANGE
BASEDIR = "../../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint

In [3]:
from src.kg_model import KnowledgeGraphModel, KnowledgeGraphModelConfig
from src.db_drivers.vector_driver import EmbedderModelConfig
from src.pipelines.memorize import MemPipelineConfig, MemPipeline, LLMUpdatorConfig

from src.pipelines.qa.kg_reasoning.weak_reasoner import WeakKGReasonerConfig, WeakKGReasoner
from src.pipelines.qa.kg_reasoning.weak_reasoner import QALLMGeneratorConfig, QueryLLMParserConfig, KnowledgeComparatorConfig, KnowledgeRetrieverConfig

from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig, MediumKGReasoner
from src.pipelines.qa.kg_reasoning.medium_reasoner import AnswerGeneratorConfig, ClueAnswerGeneratorConfig, ClueAnswersSummarizerConfig, \
    ClueQueriesGeneratorConfig, EntitiesExtractorConfig, Entities2NodesMatcherConfig, SearchPlanEnhancerConfig

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


## Пример настройки KGReasoner-стадии в рамках QA-пайплайна

1. Инициализация модели графа знаний

In [4]:
kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строиться не будет
)

EMBEDDER_MODEL_PATH = '../../../../models/intfloat/multilingual-e5-small' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_config['m-e5-small'] = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)
kg_model = KnowledgeGraphModel(kg_config)

No sentence-transformers model found with name ../../../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


2. Инициализация Memorize-пайплайна

In [5]:
mem_config = MemPipelineConfig(
    updator_config=LLMUpdatorConfig(
        delete_obsolete_info=False # Выключаем механизм по поиску/удалению устревших знаний в графе при добавлении новой информации
    )
)
mem_pipeline = MemPipeline(kg_model, mem_config)

3. Добавление информации в граф знаний

In [6]:
TEXT_EXAMPLES = [
    "Sasha was walking along the highway.", 
    "Masha was walking along the highway.", 
    "The ship was sailing along the water canal.", 
    "The motorboat was sailing along the river."]

In [7]:
extracted_triplets = []
for example in TEXT_EXAMPLES:
    tmp_extracted_triplets, _ = mem_pipeline.remember(example)
    extracted_triplets += tmp_extracted_triplets

In [8]:
kg_model.count_items()

{'graph_info': {'triplets': 31, 'nodes': 19},
 'embeddings_info': {'nodes': 19, 'triplets': 14},
 'nodestree_info': None}

4.1. Инициализация 'weak' KGReasoner-стадии

In [9]:
weak_reasoner_config = WeakKGReasonerConfig(
    query_parser_config=QueryLLMParserConfig(lang='en'),
    knowledge_comparator_config=KnowledgeComparatorConfig(),
    knowledge_retriever_config=KnowledgeRetrieverConfig(),
    answer_generator_config=QALLMGeneratorConfig(lang='en')
)
pprint(weak_reasoner_config, depth=1, width=200)

WeakKGReasonerConfig(query_parser_config=QueryLLMParserConfig(lang='en',
                                                              agent_gen_stategy=None,
                                                              kw_extraction_task_config=AgentTaskSolverConfig(version='v2',
                                                                                                              suites={...},
                                                                                                              formate_context_func=<function kwe_custom_formate at 0x7f0257861480>,
                                                                                                              postprocess_answer_func=<function kwe_custom_postprocess at 0x7f0257861510>,
                                                                                                              cache_table_name='qa_agent_kwe_task_cache',
                                                                        

In [10]:
weak_reasoner = WeakKGReasoner(kg_model, weak_reasoner_config)

In [11]:
QUERY_EXAMPLES = ["Which of the following people walked along the highway: Sasha, Masha, Katya?",
                  "Have motorboat was ever sailed through a water canal?"]

In [12]:
for query in QUERY_EXAMPLES:
    print("\nQuery: ", query)
    answer, rinfo = weak_reasoner.perform(query)
    print('Return status: ')
    pprint(rinfo)
    print("Answer: ", answer)


Query:  Which of the following people walked along the highway: Sasha, Masha, Katya?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  Sasha, Masha

Query:  Have motorboat was ever sailed through a water canal?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  <|NotEnoughtInfo|>


4.2. Инициализация 'medium' KGReasoner-стадии

In [13]:
medium_reasoner_config = MediumKGReasonerConfig(
    searchplan_enhancer_config=SearchPlanEnhancerConfig(lang='en'),
    entities_extractor_config=EntitiesExtractorConfig(lang='en'),
    e2n_matcher_config=Entities2NodesMatcherConfig(max_n=1, fetch_k=1),
    cluequeries_generator_config=ClueQueriesGeneratorConfig(lang='en'),
    knowledge_retriever_config=KnowledgeRetrieverConfig(),
    clueanswer_generator_config=ClueAnswerGeneratorConfig(lang='en'),
    clueanswers_summarizer_config=ClueAnswersSummarizerConfig(lang='en'),
    answer_generator_config=AnswerGeneratorConfig(lang='en')
)
pprint(medium_reasoner_config, depth=1, width=200)

MediumKGReasonerConfig(searchplan_enhancer_config=SearchPlanEnhancerConfig(lang='en',
                                                                           agent_gen_stategy=None,
                                                                           plan_initing_agent_task_config=AgentTaskSolverConfig(version='v1',
                                                                                                                                suites={...},
                                                                                                                                formate_context_func=<function planinit_custom_formate at 0x7f02578cb640>,
                                                                                                                                postprocess_answer_func=<function planinit_custom_postprocess at 0x7f02578cb6d0>,
                                                                                                                      

In [14]:
medium_reasoner = MediumKGReasoner(kg_model, medium_reasoner_config)

In [15]:
for query in QUERY_EXAMPLES:
    print("\nQuery: ", query)
    answer, rinfo = medium_reasoner.perform(query)
    print('Return status: ')
    pprint(rinfo)
    print("Answer: ", answer)


Query:  Which of the following people walked along the highway: Sasha, Masha, Katya?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  <final-answer>
Sasha or Masha,
because they were both alone on the highway according to Search Query #1.

Query:  Have motorboat was ever sailed through a water canal?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  Yes, motorboats have been sailed through water canals, but the exact dimensions and weight limits vary depending on the type of canal and local regulations. Typically, motorboats are designed to be compact and maneuverable for transportation through narrow waterways like canals, with maximum length and width dimensions around 20-25 feet (6-7.5 meters). However, this is highly speculative and not based on any concrete information from the finded text or search query.

However, since there is no specific example of a motorboat being sai